In [2]:
# ======================================================================
# Standalone: ChEMBL M3 ML pipeline (FP + physchem) with
#   1) Scaffold-CV benchmark: FP-only vs FP+physchem
#   2) Save final model (joblib)  ✅ picklable RDKit MorganGenerator (**Extended Connectivity FP**; ECFP)
#   3) Proper scaffold-CV error analysis (re-fit per fold)  ✅ no leakage
#      - misclassifications per compound
#      - per-scaffold performance
#      - hard scaffolds report
#
# Requirements (in openms_env):
#   - rdkit
#   - numpy, pandas
#   - scikit-learn
#   - joblib
# ======================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
)

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.DataStructs import ConvertToNumpyArray
from rdkit.Chem import Descriptors


# ----------------------------
# USER SETTINGS
# ----------------------------
DATA_PATH = r"C:\Users\Besitzer\Desktop\M3_databases\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv"

# output files
OUT_DIR = r"C:\Users\Besitzer\Desktop\M3_databases"
SAVE_MODEL_PATH = OUT_DIR + r"\m3_fp_physchem_scaffoldcv.joblib"
MISCLASS_PATH   = OUT_DIR + r"\scaffold_cv_misclassifications.csv"
SCAF_SUM_PATH   = OUT_DIR + r"\scaffold_performance_summary.csv"
HARD_SCAF_PATH  = OUT_DIR + r"\hard_scaffolds.csv"

# label mapping
POS_LABELS = {"active", "active_single"}
NEG_LABELS = {"inactive", "inactive_single"}

# sample weights (confidence weights)
WEIGHTS = {
    "active": 1.0,
    "inactive": 1.0,
    "active_single": 0.5,
    "inactive_single": 0.7,
}

# fingerprint settings
FP_RADIUS = 2
FP_NBITS = 2048
FP_USE_CHIRALITY = True
FP_USE_FEATURES = False   # <-- HIER ist der Schalter für Physchem

# scaffold CV
N_SPLITS = 10

# classifier baseline
def make_clf():
    return LogisticRegression(
        max_iter=5000,
        solver="liblinear",
        class_weight="balanced",
    )


# ======================================================================
# 1) LOAD + FILTER DATA
# ======================================================================
df = pd.read_csv(DATA_PATH)

required = {"molecule_chembl_id", "smiles", "consensus_label"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in input CSV: {missing}")

df = df[df["consensus_label"].isin(POS_LABELS | NEG_LABELS)].copy()
df = df.dropna(subset=["smiles"]).copy()
df["smiles"] = df["smiles"].astype(str).str.strip()
df = df[df["smiles"] != ""].copy()

df["y"] = df["consensus_label"].isin(POS_LABELS).astype(int)
df["w"] = df["consensus_label"].map(WEIGHTS).astype(float)

print("Loaded & filtered:", df.shape)
print(df["consensus_label"].value_counts())


# ======================================================================
# 2) SCAFFOLDS (GROUPS)
# ======================================================================
def smiles_to_scaffold(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    if scaf is None:
        return ""
    return Chem.MolToSmiles(scaf, isomericSmiles=False)

df["scaffold"] = df["smiles"].apply(smiles_to_scaffold)
bad = df["scaffold"].eq("")
if bad.any():
    print(f"Dropping {bad.sum()} rows with invalid SMILES/scaffold.")
    df = df[~bad].copy()

X_smiles = df["smiles"].values
y = df["y"].values.astype(int)
w = df["w"].values.astype(float)
groups = df["scaffold"].values

cv = GroupKFold(n_splits=N_SPLITS)


# ======================================================================
# 3) FEATURES: Morgan FP (MorganGenerator) + PhysChem
#    - IMPORTANT: MorganGenerator object is NOT picklable.
#      We make the transformer picklable by dropping _gen in __getstate__.
# ======================================================================
from rdkit.Chem import rdFingerprintGenerator

class MorganFeaturizer(BaseEstimator, TransformerMixin):
    def __init__(self, radius=2, n_bits=2048, use_chirality=True, use_features=False):
        self.radius = radius
        self.n_bits = n_bits
        self.use_chirality = use_chirality
        self.use_features = use_features
        self._gen = None  # not picklable

    def _build_gen(self):
        atom_inv_gen = None
        if self.use_features:
            # This is the FCFP-style switch:
            atom_inv_gen = rdFingerprintGenerator.GetMorganFeatureAtomInvGen()
            # feature-based invariants == FCFP-like
            # (RDKit docs: "Feature-based invariants ... similar to FCFP") :contentReference[oaicite:1]{index=1}

        self._gen = rdFingerprintGenerator.GetMorganGenerator(
            radius=self.radius,
            fpSize=self.n_bits,
            includeChirality=self.use_chirality,
            atomInvariantsGenerator=atom_inv_gen
        )

    def fit(self, X, y=None):
        self._build_gen()
        return self

    def transform(self, X):
        if self._gen is None:
            self._build_gen()

        X = np.asarray(X, dtype=object)
        feats = np.zeros((len(X), self.n_bits), dtype=np.float32)

        for i, smi in enumerate(X):
            mol = Chem.MolFromSmiles(str(smi))
            if mol is None:
                continue
            bv = self._gen.GetFingerprint(mol)
            arr = np.zeros((self.n_bits,), dtype=np.int8)
            ConvertToNumpyArray(bv, arr)
            feats[i, :] = arr
        return feats

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_gen"] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)


class PhysChemFeaturizer(BaseEstimator, TransformerMixin):
    """
    Compact physchem set (6 features):
      MolWt, MolLogP, HBD, HBA, TPSA, NumRotatableBonds
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=object)
        feats = np.zeros((len(X), 6), dtype=np.float32)

        for i, smi in enumerate(X):
            mol = Chem.MolFromSmiles(str(smi))
            if mol is None:
                feats[i, :] = np.nan
                continue
            feats[i, 0] = Descriptors.MolWt(mol)
            feats[i, 1] = Descriptors.MolLogP(mol)
            feats[i, 2] = Descriptors.NumHDonors(mol)
            feats[i, 3] = Descriptors.NumHAcceptors(mol)
            feats[i, 4] = Descriptors.TPSA(mol)
            feats[i, 5] = Descriptors.NumRotatableBonds(mol)

        # fill NaNs (rare if SMILES valid)
        if np.isnan(feats).any():
            col_med = np.nanmedian(feats, axis=0)
            r, c = np.where(np.isnan(feats))
            feats[r, c] = col_med[c]
        return feats


class ConcatFeatures(BaseEstimator, TransformerMixin):
    """Concatenate outputs of two transformers applied to the same input X."""
    def __init__(self, t1, t2):
        self.t1 = t1
        self.t2 = t2

    def fit(self, X, y=None):
        self.t1.fit(X, y)
        self.t2.fit(X, y)
        return self

    def transform(self, X):
        A = self.t1.transform(X)
        B = self.t2.transform(X)
        return np.hstack([A, B])


# ======================================================================
# 4) BUILD PIPES: FP-only vs FP+physchem
# ======================================================================
pipe_fp_only = Pipeline([
    ("feat", MorganFeaturizer(
        radius=FP_RADIUS, n_bits=FP_NBITS,
        use_chirality=FP_USE_CHIRALITY,
        use_features=FP_USE_FEATURES  # <-- jetzt aktiv
    )),
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", make_clf()),
])

pipe_fp_physchem = Pipeline([
    ("feat", ConcatFeatures(
        MorganFeaturizer(
            radius=FP_RADIUS, n_bits=FP_NBITS,
            use_chirality=FP_USE_CHIRALITY,
            use_features=FP_USE_FEATURES  # <-- jetzt aktiv
        ),
        PhysChemFeaturizer()
    )),
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", make_clf()),
])


# ======================================================================
# 5) SCAFFOLD-CV EVALUATION
# ======================================================================
def eval_pipe(pipe, name: str):
    metrics = {"roc_auc": [], "pr_auc": [], "bal_acc": [], "mcc": []}
    print(f"\n=== {name} ===")

    for fold, (tr, te) in enumerate(cv.split(X_smiles, y, groups=groups), start=1):
        X_tr, X_te = X_smiles[tr], X_smiles[te]
        y_tr, y_te = y[tr], y[te]
        w_tr = w[tr]

        pipe.fit(X_tr, y_tr, clf__sample_weight=w_tr)
        proba = pipe.predict_proba(X_te)[:, 1]
        pred = (proba >= 0.5).astype(int)

        roc = roc_auc_score(y_te, proba)
        pr  = average_precision_score(y_te, proba)
        bal = balanced_accuracy_score(y_te, pred)
        mcc = matthews_corrcoef(y_te, pred)

        metrics["roc_auc"].append(roc)
        metrics["pr_auc"].append(pr)
        metrics["bal_acc"].append(bal)
        metrics["mcc"].append(mcc)

        print(
            f"[Fold {fold}] ROC-AUC={roc:.3f} | PR-AUC={pr:.3f} | "
            f"BalAcc={bal:.3f} | MCC={mcc:.3f} | n_test={len(te)} | pos_test={int(y_te.sum())}"
        )

    print(f"\n--- {name} summary ---")
    for k, vals in metrics.items():
        vals = np.array(vals, dtype=float)
        print(f"{k}: mean={vals.mean():.3f}  std={vals.std():.3f}")

    return metrics


m1 = eval_pipe(pipe_fp_only,   "A) FP-only")
m2 = eval_pipe(pipe_fp_physchem, "B) FP + physchem")


# ======================================================================
# 6) FIT FINAL MODEL ON ALL DATA + SAVE
#    Choose FP+physchem by default (adjust if FP-only wins)
# ======================================================================
final_pipe = pipe_fp_physchem
final_pipe.fit(X_smiles, y, clf__sample_weight=w)
joblib.dump(final_pipe, SAVE_MODEL_PATH)
print("\n[SAVED MODEL]", SAVE_MODEL_PATH)


# ======================================================================
# 7) PROPER ERROR ANALYSIS (NO LEAKAGE!)
#    Re-fit a fresh clone of the pipeline within each CV fold.
# ======================================================================
rows = []
for fold, (tr, te) in enumerate(cv.split(X_smiles, y, groups=groups), start=1):
    fold_pipe = clone(final_pipe)

    X_tr, X_te = X_smiles[tr], X_smiles[te]
    y_tr, y_te = y[tr], y[te]
    w_tr = w[tr]

    fold_pipe.fit(X_tr, y_tr, clf__sample_weight=w_tr)
    proba = fold_pipe.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)

    for i, idx in enumerate(te):
        true_i = int(y_te[i])
        pred_i = int(pred[i])
        err = "OK"
        if pred_i == 1 and true_i == 0:
            err = "FP"
        elif pred_i == 0 and true_i == 1:
            err = "FN"

        rows.append({
            "fold": fold,
            "molecule_chembl_id": df.iloc[idx]["molecule_chembl_id"],
            "consensus_label": df.iloc[idx]["consensus_label"],
            "smiles": df.iloc[idx]["smiles"],
            "scaffold": df.iloc[idx]["scaffold"],
            "true_label": true_i,
            "pred_label": pred_i,
            "p_active": float(proba[i]),
            "error_type": err,
        })

err_df = pd.DataFrame(rows)
err_df.to_csv(MISCLASS_PATH, index=False)
print("\n[SAVED]", MISCLASS_PATH)
print(err_df["error_type"].value_counts())


# ======================================================================
# 8) PER-SCAFFOLD PERFORMANCE (from out-of-fold predictions)
# ======================================================================
scaf_stats = []
for scaf, g in err_df.groupby("scaffold"):
    y_true = g["true_label"].values
    y_pred = g["pred_label"].values

    # MCC undefined if only one class present within that scaffold's held-out set
    if len(np.unique(y_true)) < 2:
        continue

    scaf_stats.append({
        "scaffold": scaf,
        "n": int(len(g)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "bal_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "n_errors": int((g["error_type"] != "OK").sum()),
        "pos_frac": float(np.mean(y_true)),
    })

scaf_df = pd.DataFrame(scaf_stats).sort_values(["mcc", "n"], ascending=[True, False])
scaf_df.to_csv(SCAF_SUM_PATH, index=False)
print("[SAVED]", SCAF_SUM_PATH)
print(scaf_df.head(10))


# ======================================================================
# 9) "HARD SCAFFOLDS" REPORT
# ======================================================================
hard_scaffolds = scaf_df.query("n >= 10 and mcc < 0.5").copy()
hard_scaffolds.to_csv(HARD_SCAF_PATH, index=False)
print("\n[SAVED]", HARD_SCAF_PATH)
print("Hard scaffolds (n>=10 & MCC<0.5):", len(hard_scaffolds))
print(hard_scaffolds.head(10))


Loaded & filtered: (2268, 15)
consensus_label
active_single      1502
inactive_single     463
active              286
inactive             17
Name: count, dtype: int64

=== A) FP-only ===
[Fold 1] ROC-AUC=0.979 | PR-AUC=0.994 | BalAcc=0.905 | MCC=0.825 | n_test=227 | pos_test=184
[Fold 2] ROC-AUC=0.989 | PR-AUC=0.998 | BalAcc=0.966 | MCC=0.923 | n_test=227 | pos_test=188
[Fold 3] ROC-AUC=0.980 | PR-AUC=0.997 | BalAcc=0.873 | MCC=0.724 | n_test=227 | pos_test=199
[Fold 4] ROC-AUC=0.916 | PR-AUC=0.984 | BalAcc=0.779 | MCC=0.610 | n_test=227 | pos_test=195
[Fold 5] ROC-AUC=0.965 | PR-AUC=0.974 | BalAcc=0.930 | MCC=0.850 | n_test=227 | pos_test=192
[Fold 6] ROC-AUC=0.942 | PR-AUC=0.969 | BalAcc=0.871 | MCC=0.772 | n_test=227 | pos_test=149
[Fold 7] ROC-AUC=0.957 | PR-AUC=0.972 | BalAcc=0.828 | MCC=0.717 | n_test=227 | pos_test=140
[Fold 8] ROC-AUC=0.961 | PR-AUC=0.990 | BalAcc=0.898 | MCC=0.833 | n_test=227 | pos_test=190
[Fold 9] ROC-AUC=0.936 | PR-AUC=0.988 | BalAcc=0.850 | MCC=0.624 | n